# NB09 — Toy EVM

Build a stack-machine interpreter for ~25 EVM opcodes. Run hand-written
bytecode. Disassemble a real compiled contract's bytecode at the end.

**Exports:** `_lib/evm.py` — `EVM`, `OPCODES`, `disasm`, `MOD`.


## 1. What is the EVM?

The **Ethereum Virtual Machine** is a **stack machine** — there are no
registers. Every operation pops its inputs from the top of the stack and
pushes its result back. The full spec has ~140 opcodes; we implement ~25.

### Three memory areas

| Area | Size | Description |
|------|------|-------------|
| **Stack** | 1024 × 256-bit words | LIFO; all arithmetic happens here |
| **Memory** | byte-addressable, expands on demand | zero-initialised; used for temporary data |
| **Storage** | 256-bit key → 256-bit value | *persistent*; survives across transactions; this is how Solidity state variables work |

### Programs are bytecode

A compiled contract is a flat byte sequence. Each byte is either an
opcode (executed immediately) or push data that follows a `PUSH` opcode.
The program counter (`pc`) starts at 0 and advances one byte at a time —
or jumps when a `JUMP`/`JUMPI` instruction fires.


## 2. Opcode table (the 25 we implement)

| Op | Hex | Stack effect |
|----|-----|--------------|
| STOP | 00 | halt |
| ADD | 01 | a, b → a+b |
| MUL | 02 | a, b → a\*b |
| SUB | 03 | a, b → a-b |
| DIV | 04 | a, b → a//b (0 if b=0) |
| MOD | 06 | a, b → a%b |
| LT | 10 | a, b → 1 if a<b else 0 |
| GT | 11 | a, b → 1 if a>b else 0 |
| EQ | 14 | a, b → 1 if a==b else 0 |
| ISZERO | 15 | a → 1 if a==0 else 0 |
| AND | 16 | a, b → a&b |
| OR | 17 | a, b → a\|b |
| NOT | 19 | a → ~a (256-bit) |
| POP | 50 | a → (discard) |
| MLOAD | 51 | offset → mem[off:off+32] |
| MSTORE | 52 | offset, val → write 32 bytes |
| SLOAD | 54 | key → storage[key] |
| SSTORE | 55 | key, val → write storage |
| JUMP | 56 | dest → pc=dest |
| JUMPI | 57 | dest, cond → if cond: pc=dest |
| PC | 58 | → current pc |
| JUMPDEST | 5b | no-op marker (valid jump target) |
| PUSH1..PUSH32 | 60..7f | push N bytes of inline data |
| DUP1 | 80 | duplicate top of stack |
| SWAP1 | 90 | swap top two items |
| RETURN | f3 | offset, len → halt; return mem[off:off+len] |

> **Pop order:** `a = pop(); b = pop()`, then compute `a OP b`. For binary
> ops, `a` is the stack top (the *last* item pushed). This matches the
> Ethereum Yellow Paper convention.


## 3. Interpreter — `EVM` class

Split into four code cells to keep each manageable. Cell A: skeleton and
arithmetic opcodes.


In [1]:
MOD = 2**256  # all values are 256-bit unsigned integers


class EVM:
    def __init__(self, code: bytes):
        self.code = code
        self.pc = 0
        self.stack: list[int] = []
        self.memory = bytearray()
        self.storage: dict[int, int] = {}
        self.stopped = False
        self.return_data = b''

    def _push(self, v: int) -> None:
        if len(self.stack) >= 1024:
            raise RuntimeError('stack overflow')
        self.stack.append(v % MOD)

    def _pop(self) -> int:
        if not self.stack:
            raise RuntimeError('stack underflow')
        return self.stack.pop()

    def _expand_mem(self, end: int) -> None:
        if end > len(self.memory):
            # round up to 32-byte boundary (like the real EVM)
            pad = ((end + 31) // 32) * 32 - len(self.memory)
            self.memory.extend(b'\x00' * pad)

    def step(self) -> None:
        op = self.code[self.pc]
        self.pc += 1
        # ---- arithmetic ----
        if op == 0x00:    # STOP
            self.stopped = True
        elif op == 0x01:  # ADD
            a, b = self._pop(), self._pop(); self._push(a + b)
        elif op == 0x02:  # MUL
            a, b = self._pop(), self._pop(); self._push(a * b)
        elif op == 0x03:  # SUB
            a, b = self._pop(), self._pop(); self._push(a - b)
        elif op == 0x04:  # DIV
            a, b = self._pop(), self._pop(); self._push(0 if b == 0 else a // b)
        elif op == 0x06:  # MOD
            a, b = self._pop(), self._pop(); self._push(0 if b == 0 else a % b)
        else:
            self._step_compare(op)


print('EVM skeleton defined')


EVM skeleton defined


Cell B: comparison and bitwise opcodes — monkey-patched onto `EVM`.
Python's class system lets us split the implementation across cells
without losing any state.


In [2]:
def _step_compare(self, op: int) -> None:
    if op == 0x10:    # LT
        a, b = self._pop(), self._pop(); self._push(1 if a < b else 0)
    elif op == 0x11:  # GT
        a, b = self._pop(), self._pop(); self._push(1 if a > b else 0)
    elif op == 0x14:  # EQ
        a, b = self._pop(), self._pop(); self._push(1 if a == b else 0)
    elif op == 0x15:  # ISZERO
        a = self._pop(); self._push(1 if a == 0 else 0)
    elif op == 0x16:  # AND
        a, b = self._pop(), self._pop(); self._push(a & b)
    elif op == 0x17:  # OR
        a, b = self._pop(), self._pop(); self._push(a | b)
    elif op == 0x19:  # NOT
        a = self._pop(); self._push(~a)  # Python ~a is -(a+1); % MOD gives correct 256-bit NOT
    else:
        self._step_memory(op)


EVM._step_compare = _step_compare
print('comparisons + bitwise attached')


comparisons + bitwise attached


Cell C: stack manipulation, memory, and storage opcodes.


In [3]:
def _step_memory(self, op: int) -> None:
    if op == 0x50:    # POP
        self._pop()
    elif op == 0x51:  # MLOAD
        off = self._pop()
        self._expand_mem(off + 32)
        self._push(int.from_bytes(self.memory[off:off + 32], 'big'))
    elif op == 0x52:  # MSTORE
        off, val = self._pop(), self._pop()
        self._expand_mem(off + 32)
        self.memory[off:off + 32] = val.to_bytes(32, 'big')
    elif op == 0x54:  # SLOAD
        key = self._pop()
        self._push(self.storage.get(key, 0))
    elif op == 0x55:  # SSTORE  — first pop = key (top), second pop = value
        key, val = self._pop(), self._pop()
        self.storage[key] = val
    else:
        self._step_control(op)


EVM._step_memory = _step_memory
print('memory + storage attached')


memory + storage attached


Cell D: control flow (`JUMP`, `JUMPI`), push data, stack shuffling, and
`RETURN`. Also attaches the `run()` method.


In [4]:
def _step_control(self, op: int) -> None:
    if op == 0x56:    # JUMP
        dest = self._pop()
        if dest >= len(self.code) or self.code[dest] != 0x5b:
            raise RuntimeError(f'bad JUMP dest 0x{dest:x}')
        self.pc = dest
    elif op == 0x57:  # JUMPI
        dest, cond = self._pop(), self._pop()
        if cond != 0:
            if dest >= len(self.code) or self.code[dest] != 0x5b:
                raise RuntimeError(f'bad JUMPI dest 0x{dest:x}')
            self.pc = dest
    elif op == 0x58:  # PC
        self._push(self.pc - 1)   # pc was already incremented past the PC opcode
    elif op == 0x5b:  # JUMPDEST
        pass                       # valid jump-target marker; no-op at runtime
    elif 0x60 <= op <= 0x7f:      # PUSH1 .. PUSH32
        n = op - 0x5f             # number of data bytes to consume
        val = int.from_bytes(self.code[self.pc:self.pc + n], 'big')
        self.pc += n
        self._push(val)
    elif op == 0x80:  # DUP1
        if not self.stack:
            raise RuntimeError('stack underflow')
        self._push(self.stack[-1])
    elif op == 0x90:  # SWAP1
        if len(self.stack) < 2:
            raise RuntimeError('stack underflow')
        self.stack[-1], self.stack[-2] = self.stack[-2], self.stack[-1]
    elif op == 0xf3:  # RETURN
        off, length = self._pop(), self._pop()
        self._expand_mem(off + length)
        self.return_data = bytes(self.memory[off:off + length])
        self.stopped = True
    else:
        raise RuntimeError(f'unknown opcode 0x{op:02x} at pc={self.pc - 1}')


EVM._step_control = _step_control


def run(self, max_steps: int = 100_000) -> None:
    n = 0
    while not self.stopped and self.pc < len(self.code):
        self.step()
        n += 1
        if n >= max_steps:
            raise RuntimeError('max_steps exceeded')
    if not self.stopped:
        self.stopped = True   # ran off the end of bytecode (implicit STOP)


EVM.run = run
print('control flow + run() attached — EVM complete')


control flow + run() attached — EVM complete


## 4. Demo: `(2 + 3) * 4`

Bytecode: `PUSH1 0x04  PUSH1 0x03  PUSH1 0x02  ADD  MUL  STOP`

Let's trace every step:

```
pc=0  60 04   PUSH1 0x04   stack: [4]
pc=2  60 03   PUSH1 0x03   stack: [4, 3]
pc=4  60 02   PUSH1 0x02   stack: [4, 3, 2]   (2 on top)
pc=6  01      ADD          pop 2, pop 3 -> push 5    stack: [4, 5]
pc=7  02      MUL          pop 5, pop 4 -> push 20   stack: [20]
pc=8  00      STOP
```

Notice that `PUSH1 0x04` takes two bytes: opcode `60` plus data byte `04`.
So after the three PUSHes, `pc = 6`, not 3.


In [5]:
code = bytes.fromhex(
    '6004'   # PUSH1 0x04
    '6003'   # PUSH1 0x03
    '6002'   # PUSH1 0x02
    '01'     # ADD  -> 2+3 = 5
    '02'     # MUL  -> 5*4 = 20
    '00'     # STOP
)

evm = EVM(code)
evm.run()
assert evm.stack == [20], f'expected [20], got {evm.stack}'
print('final stack:', evm.stack)   # [20]


final stack: [20]


## 5. Storage demo — `uint x = 42`

```
PUSH1 0x2a   push 42 (value)
PUSH1 0x00   push 0  (storage slot, becomes the key)
SSTORE       storage[0] = 42
PUSH1 0x00   push 0  (same slot)
SLOAD        push storage[0] -> 42 on stack
STOP
```

This is exactly how a Solidity `uint x = 42;` works under the hood —
`SSTORE` to slot 0. Every `uint` state variable gets its own numbered
slot in the contract's persistent storage.


In [6]:
code = bytes.fromhex(
    '602a'   # PUSH1 42     (value)
    '6000'   # PUSH1 0      (key = storage slot 0)
    '55'     # SSTORE       storage[0] = 42
    '6000'   # PUSH1 0      (same slot)
    '54'     # SLOAD        push storage[0]
    '00'     # STOP
)

evm = EVM(code)
evm.run()
print('stack:  ', evm.stack)     # [42]
print('storage:', evm.storage)   # {0: 42}
assert evm.stack == [42]
assert evm.storage == {0: 42}


stack:   [42]
storage: {0: 42}


## 6. JUMPI loop — sum 1..10

Compute `sum = 1 + 2 + ... + 10 = 55` entirely in EVM bytecode.
We use storage slots as variables: slot 0 = `sum`, slot 1 = `i`.

Pseudocode:
```
storage[0] = 0   # sum
storage[1] = 1   # i
loop:
  if i > 10: goto end
  storage[0] = storage[0] + storage[1]   # sum += i
  storage[1] = storage[1] + 1            # i += 1
  goto loop
end:
  STOP
```

For the GT check: we push 10 first, then `i` (so `i` is on top). Then
`GT` pops `a=i` and `b=10` and pushes `1 if i > 10`. When `i` first
reaches 11 the JUMPI fires and we exit.

```
pc   hex              ASM                    note
0    60 00 60 00 55   PUSH1 0  PUSH1 0  SSTORE   storage[0]=0
5    60 01 60 01 55   PUSH1 1  PUSH1 1  SSTORE   storage[1]=1
10   5b               JUMPDEST              loop label
11   60 0a            PUSH1 10              10 on stack (b for GT)
13   60 01 54         PUSH1 1  SLOAD        i on top   (a for GT)
16   11               GT                    a>b = i>10?
17   60 2a 57         PUSH1 0x2a  JUMPI     if true jump to end (0x2a=42)
20   60 01 54         PUSH1 1  SLOAD        load i
23   60 00 54         PUSH1 0  SLOAD        load sum
26   01               ADD                   sum+i
27   60 00 55         PUSH1 0  SSTORE       storage[0]=sum+i
30   60 01 54         PUSH1 1  SLOAD        load i
33   60 01            PUSH1 1              
35   01               ADD                   i+1
36   60 01 55         PUSH1 1  SSTORE       storage[1]=i+1
39   60 0a 56         PUSH1 0x0a  JUMP      goto loop (0x0a=10)
42   5b               JUMPDEST              end label
43   00               STOP
```


In [7]:
# Build the bytecode programmatically so the offsets are provably correct.
loop_code = bytearray()

# ---- initialise storage ----
# SSTORE pops key (top) then value; push value first, then key.
loop_code.extend([0x60, 0x00, 0x60, 0x00, 0x55])  # storage[0] = 0 (sum)
loop_code.extend([0x60, 0x01, 0x60, 0x01, 0x55])  # storage[1] = 1 (i)

LOOP = len(loop_code)   # pc=10 — loop JUMPDEST
loop_code.extend([0x5b])                           # JUMPDEST loop

# ---- condition: i > 10? ----
# Push 10 first (b), then i (a); GT pops a then b -> 1 if a>b = i>10
loop_code.extend([0x60, 0x0a])                     # PUSH1 10
loop_code.extend([0x60, 0x01, 0x54])               # PUSH1 slot1, SLOAD -> i
loop_code.extend([0x11])                            # GT  (i > 10?)

JUMPI_ARG = len(loop_code) + 1  # index of the end-dest byte (filled later)
loop_code.extend([0x60, 0x00, 0x57])               # PUSH1 <end>, JUMPI  (placeholder)

# ---- loop body ----
loop_code.extend([0x60, 0x01, 0x54])               # load i
loop_code.extend([0x60, 0x00, 0x54])               # load sum
loop_code.extend([0x01])                            # ADD -> sum+i
loop_code.extend([0x60, 0x00, 0x55])               # storage[0] = sum+i
loop_code.extend([0x60, 0x01, 0x54])               # load i
loop_code.extend([0x60, 0x01])                     # PUSH1 1
loop_code.extend([0x01])                            # ADD -> i+1
loop_code.extend([0x60, 0x01, 0x55])               # storage[1] = i+1
loop_code.extend([0x60, LOOP, 0x56])               # PUSH1 <loop>, JUMP

# ---- end label ----
END = len(loop_code)    # pc=42
loop_code[JUMPI_ARG] = END                          # back-patch the JUMPI target
loop_code.extend([0x5b])                            # JUMPDEST end
loop_code.extend([0x00])                            # STOP

loop_code = bytes(loop_code)
print(f'bytecode ({len(loop_code)} bytes): {loop_code.hex()}')
print(f'LOOP dest=0x{LOOP:02x}  END dest=0x{END:02x}')
assert loop_code[LOOP] == 0x5b, 'loop JUMPDEST missing'
assert loop_code[END]  == 0x5b, 'end  JUMPDEST missing'

evm = EVM(loop_code)
evm.run()
assert evm.storage[0] == 55, f'sum was {evm.storage[0]}, expected 55'
print('sum 1..10 =', evm.storage[0])   # 55


bytecode (44 bytes): 600060005560016001555b600a60015411602a5760015460005401600055600154600101600155600a565b00
LOOP dest=0x0a  END dest=0x2a
sum 1..10 = 55


## 7. Disassembler

A disassembler reads bytecode and prints human-readable assembly. The
tricky part is `PUSH` instructions: after each `PUSH1`..`PUSH32` opcode
byte, there are `N` data bytes that must be consumed before the next
opcode. We extend the opcode table with a few hundred real Ethereum
opcodes so real contract bytecode prints cleanly.


In [8]:
OPCODES: dict[int, str] = {
    # ---- arithmetic ----
    0x00: 'STOP',  0x01: 'ADD',    0x02: 'MUL',  0x03: 'SUB',
    0x04: 'DIV',   0x05: 'SDIV',   0x06: 'MOD',  0x07: 'SMOD',
    0x08: 'ADDMOD',0x09: 'MULMOD', 0x0a: 'EXP',  0x0b: 'SIGNEXTEND',
    # ---- comparison / bitwise ----
    0x10: 'LT',    0x11: 'GT',    0x12: 'SLT',  0x13: 'SGT',
    0x14: 'EQ',    0x15: 'ISZERO',0x16: 'AND',  0x17: 'OR',
    0x18: 'XOR',   0x19: 'NOT',   0x1a: 'BYTE', 0x1b: 'SHL',
    0x1c: 'SHR',   0x1d: 'SAR',
    # ---- hashing / env ----
    0x20: 'KECCAK256',
    0x30: 'ADDRESS',   0x31: 'BALANCE',    0x32: 'ORIGIN',
    0x33: 'CALLER',    0x34: 'CALLVALUE',  0x35: 'CALLDATALOAD',
    0x36: 'CALLDATASIZE', 0x37: 'CALLDATACOPY', 0x38: 'CODESIZE',
    0x39: 'CODECOPY', 0x3a: 'GASPRICE',   0x3b: 'EXTCODESIZE',
    0x3c: 'EXTCODECOPY', 0x3d: 'RETURNDATASIZE', 0x3e: 'RETURNDATACOPY',
    0x3f: 'EXTCODEHASH',
    0x40: 'BLOCKHASH', 0x41: 'COINBASE',  0x42: 'TIMESTAMP',
    0x43: 'NUMBER',    0x44: 'DIFFICULTY',0x45: 'GASLIMIT',
    0x46: 'CHAINID',   0x47: 'SELFBALANCE',0x48: 'BASEFEE',
    # ---- memory / storage / control ----
    0x50: 'POP',   0x51: 'MLOAD',  0x52: 'MSTORE', 0x53: 'MSTORE8',
    0x54: 'SLOAD', 0x55: 'SSTORE', 0x56: 'JUMP',   0x57: 'JUMPI',
    0x58: 'PC',    0x59: 'MSIZE',  0x5a: 'GAS',    0x5b: 'JUMPDEST',
    # ---- DUP1..DUP16, SWAP1..SWAP16 ----
    **{0x80 + i: f'DUP{i + 1}'  for i in range(16)},
    **{0x90 + i: f'SWAP{i + 1}' for i in range(16)},
    # ---- logging ----
    0xa0: 'LOG0', 0xa1: 'LOG1', 0xa2: 'LOG2', 0xa3: 'LOG3', 0xa4: 'LOG4',
    # ---- system ----
    0xf0: 'CREATE',       0xf1: 'CALL',         0xf2: 'CALLCODE',
    0xf3: 'RETURN',       0xf4: 'DELEGATECALL', 0xf5: 'CREATE2',
    0xfa: 'STATICCALL',   0xfd: 'REVERT',       0xfe: 'INVALID',
    0xff: 'SELFDESTRUCT',
}


def disasm(code: bytes) -> list[str]:
    """Disassemble EVM bytecode into a list of human-readable lines."""
    out, pc = [], 0
    while pc < len(code):
        op = code[pc]
        if 0x60 <= op <= 0x7f:          # PUSH1 .. PUSH32
            n = op - 0x5f
            data = code[pc + 1:pc + 1 + n].hex()
            out.append(f'{pc:04x}  PUSH{n} 0x{data}')
            pc += 1 + n
        else:
            name = OPCODES.get(op, f'?? (0x{op:02x})')
            out.append(f'{pc:04x}  {name}')
            pc += 1
    return out


print('OPCODES and disasm defined')
# Sanity: disassemble our loop bytecode
print('\nDisassembly of loop_code:')
for line in disasm(loop_code):
    print(' ', line)


OPCODES and disasm defined

Disassembly of loop_code:
  0000  PUSH1 0x00
  0002  PUSH1 0x00
  0004  SSTORE
  0005  PUSH1 0x01
  0007  PUSH1 0x01
  0009  SSTORE
  000a  JUMPDEST
  000b  PUSH1 0x0a
  000d  PUSH1 0x01
  000f  SLOAD
  0010  GT
  0011  PUSH1 0x2a
  0013  JUMPI
  0014  PUSH1 0x01
  0016  SLOAD
  0017  PUSH1 0x00
  0019  SLOAD
  001a  ADD
  001b  PUSH1 0x00
  001d  SSTORE
  001e  PUSH1 0x01
  0020  SLOAD
  0021  PUSH1 0x01
  0023  ADD
  0024  PUSH1 0x01
  0026  SSTORE
  0027  PUSH1 0x0a
  0029  JUMP
  002a  JUMPDEST
  002b  STOP


## 8. Read REAL deployed bytecode

Foundry writes compiled artifacts to `out/<Contract>.sol/<Contract>.json`.
The `deployedBytecode.object` field holds the hex-encoded bytecode that
lives at the contract's address on-chain.


In [9]:
import json, pathlib

art_path = pathlib.Path('../blockchain_primer/out/Counter.sol/Counter.json')
if not art_path.exists():
    print(f'WARNING: {art_path} not found — using hardcoded fallback.')
    # ERC-20-like selector dispatcher prefix; illustrative only
    deployed = bytes.fromhex('608060405234801561001057600080fd5b50')
else:
    art = json.loads(art_path.read_text())
    deployed_hex = art['deployedBytecode']['object'].removeprefix('0x')
    deployed = bytes.fromhex(deployed_hex)
    print('loaded from Counter.json')

print(f'deployed bytecode: {len(deployed)} bytes')
print('\nfirst 50 instructions:')
for line in disasm(deployed)[:50]:
    print(' ', line)


loaded from Counter.json
deployed bytecode: 279 bytes

first 50 instructions:
  0000  PUSH1 0x80
  0002  PUSH1 0x40
  0004  MSTORE
  0005  CALLVALUE
  0006  DUP1
  0007  ISZERO
  0008  PUSH1 0x0e
  000a  JUMPI
  000b  ?? (0x5f)
  000c  DUP1
  000d  REVERT
  000e  JUMPDEST
  000f  POP
  0010  PUSH1 0x04
  0012  CALLDATASIZE
  0013  LT
  0014  PUSH1 0x3a
  0016  JUMPI
  0017  ?? (0x5f)
  0018  CALLDATALOAD
  0019  PUSH1 0xe0
  001b  SHR
  001c  DUP1
  001d  PUSH4 0x26e1dba3
  0022  EQ
  0023  PUSH1 0x3e
  0025  JUMPI
  0026  DUP1
  0027  PUSH4 0x8381f58a
  002c  EQ
  002d  PUSH1 0x46
  002f  JUMPI
  0030  DUP1
  0031  PUSH4 0xd09de08a
  0036  EQ
  0037  PUSH1 0x5f
  0039  JUMPI
  003a  JUMPDEST
  003b  ?? (0x5f)
  003c  DUP1
  003d  REVERT
  003e  JUMPDEST
  003f  PUSH1 0x44
  0041  PUSH1 0x65
  0043  JUMP
  0044  JUMPDEST
  0045  STOP
  0046  JUMPDEST
  0047  PUSH1 0x4d
  0049  ?? (0x5f)


### What you just read

You are looking at the actual EVM instructions Solidity emitted for
`Counter.sol`. The pattern at the top of every contract is the
**selector dispatcher**:

1. Load the first 4 bytes of calldata (`CALLDATALOAD` + shift) — these
   are the **function selector**: the first 4 bytes of
   `keccak256("functionName(paramTypes)")`.
2. Compare the selector to each known selector via `EQ`.
3. `JUMPI` to the matching function body.

Our toy EVM cannot *run* this contract — we are missing `CALLDATALOAD`,
`KECCAK256`, `CALL`, `LOG`, `REVERT`, `CREATE`, and about 110 other
opcodes. But you can **read** it. That is a real, practical skill: when
something goes wrong on-chain, this is what engineers look at.


## 9. What is not here

The ~115 opcodes we skipped include:

- **`CALL` / `DELEGATECALL` / `CREATE` / `CREATE2`** — cross-contract
  calls; the engine for composability and proxy patterns.
- **`LOG0`..`LOG4`** — emit events that off-chain listeners consume.
- **`REVERT`** — roll back state changes and return an error message.
- **Gas accounting** — every opcode costs gas; the EVM halts when gas
  runs out. We didn't track `gas` or charge anything.
- **`KECCAK256`** — in-EVM hashing; used by mappings and the selector
  dispatcher.
- **Precompiles** — built-in contracts at addresses 0x01..0x0a for
  ECDSA recovery, hashing, BN128 pairing, etc.
- **`PUSH0`** (0x5f, added in Shanghai fork) — push a zero byte cheaply.

The spine is built. From here, `notebooks/blockchain_primer/` takes over:
real Solidity contracts, Foundry tests, and a live local chain.


## 10. Export `_lib/evm.py`

Write all reusable symbols to `_lib/evm.py` so future notebooks can
`from _lib.evm import EVM, disasm`.


In [10]:
%%writefile _lib/evm.py
"""evm.py — toy EVM interpreter (~25 opcodes).

Stack machine interpreter for a subset of the Ethereum Virtual Machine.
Supports arithmetic, comparisons, bitwise ops, memory, storage, JUMP/JUMPI,
PUSH/DUP/SWAP, and RETURN. Does not implement gas accounting.

Exported symbols: MOD, EVM, OPCODES, disasm
"""

MOD = 2**256  # all values are 256-bit unsigned integers


class EVM:
    def __init__(self, code: bytes):
        self.code = code
        self.pc = 0
        self.stack: list[int] = []
        self.memory = bytearray()
        self.storage: dict[int, int] = {}
        self.stopped = False
        self.return_data = b''

    def _push(self, v: int) -> None:
        if len(self.stack) >= 1024:
            raise RuntimeError('stack overflow')
        self.stack.append(v % MOD)

    def _pop(self) -> int:
        if not self.stack:
            raise RuntimeError('stack underflow')
        return self.stack.pop()

    def _expand_mem(self, end: int) -> None:
        if end > len(self.memory):
            pad = ((end + 31) // 32) * 32 - len(self.memory)
            self.memory.extend(b'\x00' * pad)

    def step(self) -> None:
        op = self.code[self.pc]
        self.pc += 1
        if op == 0x00:
            self.stopped = True
        elif op == 0x01:
            a, b = self._pop(), self._pop(); self._push(a + b)
        elif op == 0x02:
            a, b = self._pop(), self._pop(); self._push(a * b)
        elif op == 0x03:
            a, b = self._pop(), self._pop(); self._push(a - b)
        elif op == 0x04:
            a, b = self._pop(), self._pop(); self._push(0 if b == 0 else a // b)
        elif op == 0x06:
            a, b = self._pop(), self._pop(); self._push(0 if b == 0 else a % b)
        else:
            self._step_compare(op)

    def _step_compare(self, op: int) -> None:
        if op == 0x10:
            a, b = self._pop(), self._pop(); self._push(1 if a < b else 0)
        elif op == 0x11:
            a, b = self._pop(), self._pop(); self._push(1 if a > b else 0)
        elif op == 0x14:
            a, b = self._pop(), self._pop(); self._push(1 if a == b else 0)
        elif op == 0x15:
            a = self._pop(); self._push(1 if a == 0 else 0)
        elif op == 0x16:
            a, b = self._pop(), self._pop(); self._push(a & b)
        elif op == 0x17:
            a, b = self._pop(), self._pop(); self._push(a | b)
        elif op == 0x19:
            a = self._pop(); self._push(~a)
        else:
            self._step_memory(op)

    def _step_memory(self, op: int) -> None:
        if op == 0x50:
            self._pop()
        elif op == 0x51:
            off = self._pop()
            self._expand_mem(off + 32)
            self._push(int.from_bytes(self.memory[off:off + 32], 'big'))
        elif op == 0x52:
            off, val = self._pop(), self._pop()
            self._expand_mem(off + 32)
            self.memory[off:off + 32] = val.to_bytes(32, 'big')
        elif op == 0x54:
            key = self._pop()
            self._push(self.storage.get(key, 0))
        elif op == 0x55:
            key, val = self._pop(), self._pop()
            self.storage[key] = val
        else:
            self._step_control(op)

    def _step_control(self, op: int) -> None:
        if op == 0x56:
            dest = self._pop()
            if dest >= len(self.code) or self.code[dest] != 0x5b:
                raise RuntimeError(f'bad JUMP dest 0x{dest:x}')
            self.pc = dest
        elif op == 0x57:
            dest, cond = self._pop(), self._pop()
            if cond != 0:
                if dest >= len(self.code) or self.code[dest] != 0x5b:
                    raise RuntimeError(f'bad JUMPI dest 0x{dest:x}')
                self.pc = dest
        elif op == 0x58:
            self._push(self.pc - 1)
        elif op == 0x5b:
            pass
        elif 0x60 <= op <= 0x7f:
            n = op - 0x5f
            val = int.from_bytes(self.code[self.pc:self.pc + n], 'big')
            self.pc += n
            self._push(val)
        elif op == 0x80:
            if not self.stack:
                raise RuntimeError('stack underflow')
            self._push(self.stack[-1])
        elif op == 0x90:
            if len(self.stack) < 2:
                raise RuntimeError('stack underflow')
            self.stack[-1], self.stack[-2] = self.stack[-2], self.stack[-1]
        elif op == 0xf3:
            off, length = self._pop(), self._pop()
            self._expand_mem(off + length)
            self.return_data = bytes(self.memory[off:off + length])
            self.stopped = True
        else:
            raise RuntimeError(f'unknown opcode 0x{op:02x} at pc={self.pc - 1}')

    def run(self, max_steps: int = 100_000) -> None:
        n = 0
        while not self.stopped and self.pc < len(self.code):
            self.step()
            n += 1
            if n >= max_steps:
                raise RuntimeError('max_steps exceeded')
        if not self.stopped:
            self.stopped = True


OPCODES: dict[int, str] = {
    0x00: 'STOP',  0x01: 'ADD',    0x02: 'MUL',  0x03: 'SUB',
    0x04: 'DIV',   0x05: 'SDIV',   0x06: 'MOD',  0x07: 'SMOD',
    0x08: 'ADDMOD',0x09: 'MULMOD', 0x0a: 'EXP',  0x0b: 'SIGNEXTEND',
    0x10: 'LT',    0x11: 'GT',    0x12: 'SLT',  0x13: 'SGT',
    0x14: 'EQ',    0x15: 'ISZERO',0x16: 'AND',  0x17: 'OR',
    0x18: 'XOR',   0x19: 'NOT',   0x1a: 'BYTE', 0x1b: 'SHL',
    0x1c: 'SHR',   0x1d: 'SAR',
    0x20: 'KECCAK256',
    0x30: 'ADDRESS',   0x31: 'BALANCE',    0x32: 'ORIGIN',
    0x33: 'CALLER',    0x34: 'CALLVALUE',  0x35: 'CALLDATALOAD',
    0x36: 'CALLDATASIZE', 0x37: 'CALLDATACOPY', 0x38: 'CODESIZE',
    0x39: 'CODECOPY', 0x3a: 'GASPRICE',   0x3b: 'EXTCODESIZE',
    0x3c: 'EXTCODECOPY', 0x3d: 'RETURNDATASIZE', 0x3e: 'RETURNDATACOPY',
    0x3f: 'EXTCODEHASH',
    0x40: 'BLOCKHASH', 0x41: 'COINBASE',  0x42: 'TIMESTAMP',
    0x43: 'NUMBER',    0x44: 'DIFFICULTY',0x45: 'GASLIMIT',
    0x46: 'CHAINID',   0x47: 'SELFBALANCE',0x48: 'BASEFEE',
    0x50: 'POP',   0x51: 'MLOAD',  0x52: 'MSTORE', 0x53: 'MSTORE8',
    0x54: 'SLOAD', 0x55: 'SSTORE', 0x56: 'JUMP',   0x57: 'JUMPI',
    0x58: 'PC',    0x59: 'MSIZE',  0x5a: 'GAS',    0x5b: 'JUMPDEST',
    **{0x80 + i: f'DUP{i + 1}'  for i in range(16)},
    **{0x90 + i: f'SWAP{i + 1}' for i in range(16)},
    0xa0: 'LOG0', 0xa1: 'LOG1', 0xa2: 'LOG2', 0xa3: 'LOG3', 0xa4: 'LOG4',
    0xf0: 'CREATE',       0xf1: 'CALL',         0xf2: 'CALLCODE',
    0xf3: 'RETURN',       0xf4: 'DELEGATECALL', 0xf5: 'CREATE2',
    0xfa: 'STATICCALL',   0xfd: 'REVERT',       0xfe: 'INVALID',
    0xff: 'SELFDESTRUCT',
}


def disasm(code: bytes) -> list[str]:
    """Disassemble EVM bytecode into a list of human-readable lines."""
    out, pc = [], 0
    while pc < len(code):
        op = code[pc]
        if 0x60 <= op <= 0x7f:
            n = op - 0x5f
            data = code[pc + 1:pc + 1 + n].hex()
            out.append(f'{pc:04x}  PUSH{n} 0x{data}')
            pc += 1 + n
        else:
            name = OPCODES.get(op, f'?? (0x{op:02x})')
            out.append(f'{pc:04x}  {name}')
            pc += 1
    return out


Writing _lib/evm.py


In [11]:
import importlib, sys
# Force a clean reload in case evm.py was imported in a previous run
if '_lib.evm' in sys.modules:
    del sys.modules['_lib.evm']

import _lib.evm as _evm

# Smoke-test: arithmetic demo via the imported class
_e = _evm.EVM(bytes.fromhex('6004600360020102' + '00'))
_e.run()
assert _e.stack == [20], f'lib smoke-test failed: {_e.stack}'

# Verify disasm works on loop_code
lines = _evm.disasm(loop_code)
assert len(lines) > 0

print('_lib/evm.py: EVM, OPCODES, disasm, MOD all present and working')


_lib/evm.py: EVM, OPCODES, disasm, MOD all present and working
